### Query Translation - HYDE
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some way as to improve retrival.
* **Query re-writing** is one approach - we discussed techniques, such as _RAG Fusion_ and _Multi Query_ in previous notebooks. The idea there is to essentially modify the user-query (generate various variations of the query), so we can capture additional perspectives of what the user intends to ask.
* **Query Decomposition** is another approach, where we break down a complex query into various sub-queries (or sub-questions) which are then independently executed against the LLM and results are then combined together.

### What is HYDE?
HYDE is an interesting approach that takes adbantage of a very simple idea. The basic RAG flow takes a question and embeds it; takes a document & embeds it and looks for similarity between an embeded document & the embedded question. However, the question & document are very dis-similar. A document can very large and complex - may come from _dense_ publications (such as PDFs) and other sources, whereas questions are usually short & terse and could be ill-worded from users.

The intuition behind HYDE is take questions and map them into document space using a hypothetical document (or by generating a hypothetical document)
 
![HYDE](images/hyde.png)

#### Examples
| User Question                                     | Step-back Question                                |
|:--------------------------------------------------|:--------------------------------------------------|
| In which country was Sardar Vallabhai Patel born? | What is Sardar Vallabhai Patel's personal history |


In [15]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from IPython.display import display, Markdown


from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# since we are using Gemini, we'll use Google embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [2]:
# load API keys from .env files
load_dotenv(override=True)
# for colorful text output
console = Console()

In [3]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
faiss_store = pathlib.Path(os.getcwd()) / "faiss_index_rag_qd"

In [4]:
def create_or_load_embeddings():
    """creates if not available or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            chunk_size=300, chunk_overlap=50
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        # Use a Gemini embedding model that is suitable for retrieval.
        # It is important to match the model to the task.
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [5]:
retriever = create_or_load_embeddings()

Loading existing embeddings from 
c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\faiss_index_rag_qd

In [7]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

In [10]:
generate_queries_step_back = prompt | llm | StrOutputParser()
question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})

'How do LLM agents approach complex problems?'

In [16]:
# Response prompt
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,
        # Pass on the question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

response = chain.invoke({"question": question})
print(display(Markdown(response)))

Task decomposition for LLM agents is a crucial planning component where the agent breaks down large, complex tasks into smaller, more manageable subgoals or steps. This process enables the agent to handle intricate problems more efficiently.

Key aspects of task decomposition include:
*   **Purpose**: To transform a big, complicated task into multiple simpler, manageable steps, making the overall task easier for the LLM to process and execute.
*   **Techniques**:
    *   **Chain of Thought (CoT)**: A standard prompting technique that instructs the model to "think step by step," allowing it to decompose hard tasks and utilize more computation time. CoT also provides insight into the model's thinking process.
    *   **Tree of Thoughts (ToT)**: An extension of CoT that explores multiple reasoning possibilities at each step. It decomposes a problem into multiple thought steps and generates several thoughts per step, forming a tree structure. Search processes like Breadth-First Search (BFS) or Depth-First Search (DFS) can be used, with each state evaluated by a classifier or majority vote.
*   **Methods of Implementation**:
    *   **LLM with simple prompting**: The LLM itself can perform decomposition using prompts like "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?".
    *   **Task-specific instructions**: Providing specific instructions tailored to the task, such as "Write a story outline" for novel writing.
    *   **Human inputs**: Incorporating human guidance to define the decomposition steps.

By breaking down tasks into subgoals, LLM agents can effectively plan ahead and improve their performance on complex assignments.

None
